**Table of contents**<a id='toc0_'></a>    
- 1. [模型压缩方法介绍](#toc1_)    
  - 1.1. [模型压缩简介](#toc1_1_)    
- 2. [模型剪枝、量化与知识蒸馏](#toc2_)    
  - 2.1. [剪枝概念](#toc2_1_)    
  - 2.2. [剪枝方式分类](#toc2_2_)    
  - 2.3. [量化（学术界）](#toc2_3_)    
  - 2.4. [量化（工业界）](#toc2_4_)    
- 3. [知识蒸馏在大模型中的应用](#toc3_)    
- 4. [如何使用知识蒸馏法快速训练大模型](#toc4_)    
- 5. [案例：基于Qwen模型的知识蒸馏案例](#toc5_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[模型压缩方法介绍](#toc0_)

## 1.1. <a id='toc1_1_'></a>[模型压缩简介](#toc0_)

深度学习（Deep Learning）因其计算复杂度或参数冗余，在一些场景和设备上限制了相应的模型部署，需要借助模型压缩、优化加速、异构计算等方法突破瓶颈。

模型压缩算法能够有效降低参数冗余，从而减少存储占用、通信带宽和计算复杂度，有助于深度学习的应用部署，具体可划分为如下几种方法（后续重点介绍剪枝与量化）：

①线性或非线性量化：1/2bits, int8 和 fp16等；  
②结构或非结构剪枝：deep compression, channel pruning 和 network slimming等；  
③知识蒸馏与网络结构简化（squeeze-net, mobile-net, shuffle-net）等；  


# 2. <a id='toc2_'></a>[模型剪枝、量化与知识蒸馏](#toc0_)

## 2.1. <a id='toc2_1_'></a>[剪枝概念](#toc0_)

模型量化是指通过减少权重表示或激活所需的比特数来压缩模型；  
模型剪枝研究模型权重中的冗余，并尝试删除/修剪冗余和非关键的权重。
模型剪枝并没有很好的商业化应用，原因是剪枝的结果不可控，你不知道被剪掉的参数是否是核心参数。

<img src="./Image/2025-05-05-23-05-53.png" style="margin-left: 0" width="80%">


## 2.2. <a id='toc2_2_'></a>[剪枝方式分类](#toc0_)

Ø 非结构剪枝：通常是连接级、细粒度的剪枝方法，精度相对较高，但依赖于特定算法库或硬件平台的支持

Ø 结构剪枝：是filter级或layer级、粗粒度的剪枝方法，精度相对较低，但剪枝策略更为有效，不需要特定算法库或硬件平台的支持，能够直接在成熟深度学习框架上运行

Ø 局部方式的、通过layer by layer方式的、最小化输出FM重建误差的Channel Pruning,ThiNet Discrimination-aware Channel Pruning ；

Ø 全局方式的、通过训练期间对BN层Gamma系数施加L1正则约束的Network Slimming 


![](Image/2025-05-05-23-18-41.png)

## 2.3. <a id='toc2_3_'></a>[量化（学术界）](#toc0_)

 低精度（Low precision）可能是最通用的概念。常规精度一般使用 FP32（32位浮点，单精度）存储模型权重；低精度则表示 FP16（半精度浮点），INT8（8位的定点整数）等等数值格式。不过目前低精度往往指代 INT8。

Ø 混合精度（Mixed precision）在模型中使用 FP32 和 FP16 。 FP16 减少了一半的内存大小，但有些参数或操作符必须采用 FP32 格式才能保持准确度。如果您对该主题感兴趣，请查看 Mixed-Precision Training of Deep Neural Networks 。

Ø 量化一般指 INT8 。

Ø 根据存储一个权重元素所需的位数，还可以包括：

①二值神经网络：在运行时权重和激活只取两种值（例如 +1，-1）的神经网络，以及在训练时计算参数的梯度。

②三元权重网络：权重约束为+1,0和-1的神经网络。

③XNOR网络：过滤器和卷积层的输入是二进制的。 XNOR 网络主要使用二进制运算来近似卷积。


## 2.4. <a id='toc2_4_'></a>[量化（工业界）](#toc0_)

理论是一回事，实践是另一回事。如果一种技术方法难以推广到通用场景，则需要进行大量的额外支持。花哨的研究往往是过于棘手或前提假设过强，以至几乎无法引入工业界的软件栈。

工业界最终选择了 INT8 量化—— FP32 在推理（inference）期间被 INT8 取代，而训练（training）仍然是 FP32。TensorRT，TensorFlow，PyTorch，MxNet 和许多其他深度学习软件都已启用（或正在启用）量化。

通常，可以根据 FP32 和 INT8 的转换机制对解决方案进行分类。一些框架简单地引入了 Quantize 和 Dequantize 层，当从卷积或全链接层送入或取出时，它将 FP32 转换为INT8 或相反。在这种情况下，如图四的上半部分所示，模型本身和输入/输出采用 FP32 格式。深度学习框架加载模型，重写网络以插入Quantize 和 Dequantize 层，并将权重转换为 INT8 格式。


# 3. <a id='toc3_'></a>[知识蒸馏在大模型中的应用](#toc0_)

# 4. <a id='toc4_'></a>[如何使用知识蒸馏法快速训练大模型](#toc0_)

## 总体逻辑

有套数据（Image Batch），同时给到两个模型（Teacher Network和Student Network），两个模型分别得到两个损失，用Teacher模型的损失优化Student模型损失。

![](Image/2025-05-06-22-32-45.png)

# 5. <a id='toc5_'></a>[案例：基于Qwen模型的知识蒸馏案例](#toc0_)

<img src="./Image/2025-04-29-23-20-43.png" style="margin-left: 0" width="50%">
